In [1]:
import sys
import openai
import gradio

print(sys.executable)
print("OpenAI and Gradio imports work")

/Users/kshitijgurung/Desktop/miu/AI Engineering/lab2/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/Users/kshitijgurung/Desktop/miu/AI Engineering/lab2/.venv/bin/python
OpenAI and Gradio imports work


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("Working directory:", Path.cwd())
print("API Key:", bool(os.getenv("OPENAI_API_KEY")))

Working directory: /Users/kshitijgurung/Desktop/miu/AI Engineering/lab2
API Key: True


In [11]:
from openai import OpenAI

client = OpenAI()

SYSTEM_PROMPT = """
You are an academic study-planning assistant.
Give practical, concise study advice.
If essential details are missing, ask one clarifying question.
Do not claim to know a student's course requirements or exam content unless they provide them.
"""

def study_assistant(user_message, history = None):
    exception_msg = "I couldn't reach the AI service. Please try again."

    if not isinstance(user_message, str) or not user_message.strip():
        return "Please enter a study question."

    user_message = user_message.strip()

    if len(user_message) > 1000:
        return "Please keep your question under 1,000 characters."

    messages = []

    for turn in (history or [])[-10: ]:
        if not isinstance(turn, dict) or turn.get("role") not in ("user", "assistant"):
            continue

        content = turn.get("content")

        if isinstance(content, str):
            text = content
        elif isinstance(content, list):
            text = " ".join(
                block["text"]
                for block in content
                if isinstance(block, dict)
                and block.get("type") == "text"
                and isinstance(block.get("text"), str)
            )
        else:
            continue

        if text.strip():
            messages.append({
                "role": turn["role"],
                "content": text.strip(),
            })

    messages.append({"role" : "user", "content" : user_message})

    try:
        response = client.responses.create(
            model="gpt-6-luna",
            instructions=SYSTEM_PROMPT,
            input=messages,
            )

        return response.output_text
    except Exception:
        return exception_msg

In [13]:
print(study_assistant(""))
print(study_assistant("How should I study Python lists in one evening?"))
print(study_assistant("a" * 1001))

Please enter a study question.
Here’s a practical one-evening plan—adjust the times to fit your schedule:

1. **Learn the basics (25 min):** Make lists, access items with indexes, use negative indexes, and get slices.
2. **Practice changing lists (25 min):** Try assignment, `append`, `extend`, `insert`, `remove`, `pop`, and `sort`. Notice which methods change the list and what they return.
3. **Loop and work with lists (30 min):** Write `for` loops that count, filter, or transform items. Try a simple list comprehension.
4. **Check tricky behavior (15 min):** Explore what happens when you modify a list while looping, and the difference between `b = a` and `b = a.copy()`.
5. **Do a small coding challenge (30–45 min):** For example, take a list of numbers and produce its sum, largest value, and a new list containing only the even numbers—without using shortcuts you haven’t practiced.
6. **Review from memory (10 min):** Close your notes and explain indexing, slicing, common methods, and co

# StudyMate AI: Academic Study-Planning Chatbot

## Overview
StudyMate AI is an LLM-powered chatbot that helps students create practical study plans, review learning topics, and prepare for exams.

## Features
- Personalized study-planning guidance
- Input validation for empty and overly long messages
- LLM-powered answers using the OpenAI Responses API
- User-friendly Gradio chatbot interface
- Safe error handling for unavailable API services

## Workflow
1. The user enters a study-related question.
2. The application validates the input.
3. The input and system prompt are sent to the OpenAI model.
4. The chatbot returns concise academic guidance.
5. The Gradio interface displays the conversation.

In [15]:
import gradio as gr

def chat_with_studymate(message, history):
    return study_assistant(message, history)

with gr.Blocks(title="StudyMate AI") as demo:
    gr.Markdown("""
    # 📚 StudyMate AI
    ### Your AI-powered academic study-planning assistant

    Ask questions about study schedules, exam preparation, programming topics,
    time management, or learning strategies.
    """)

    chatbot = gr.ChatInterface(
        fn = chat_with_studymate,
        examples=[
            "Create a 5-day study plan for a Java exam.",
            "How can I prepare for a database exam this weekend?",
            "I have 2 hours tonight. How should I study Python functions?",
            "Help me create a study schedule for three subjects."
        ],
        textbox=gr.Textbox(
            placeholder="Type your study question here...",
            label="Your study question",
            lines=1,
            submit_btn=True,
            stop_btn=True
        ),
        chatbot=gr.Chatbot(
            height=420,
            placeholder="Hi! I am StudyMate AI. What would you like help studying today?"
        )
    )

    demo.launch(share=True, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://df70a409b4c5fd1fd5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Testing and validation

| Test | Input | Expected result | Result |
|---|---|---|---|
| Empty input | `""` | Ask the user to enter a question | Passed |
| Valid question | `How should I study Python lists in one evening?` | Return a practical study plan | Passed |
| Long input | 1,001 characters | Reject input over 1,000 characters | Passed |
| Follow-up question | Ask for a plan, then ask to expand one day | Use the earlier plan as context | Passed |

The chatbot validates input before sending it to the LLM. I also tested a two-message conversation in the Gradio interface to verify that follow-up questions use the previous exchange.

## Gradio interface

The interface provides example questions, a message box, and a conversation view. The screenshot below shows StudyMate responding to a study-planning question.

![StudyMate AI chatbot interface](studymate_ui.png)